# Задача 2. Iterative

Обе функции сортируют целые числа по возрастанию и возвращают новый список. Входной список остаётся прежним. Ни в одной функции нет рекурсивных вызовов.

## `merge_sort_iterative`: слияние снизу вверх

Одиночный элемент уже отсортирован. На первом проходе алгоритм сливает соседние одиночные элементы, на втором — отсортированные отрезки длиной два и так далее. Последний отрезок может быть короче остальных.

При слиянии функция сравнивает первые ещё не записанные элементы двух отрезков и кладёт меньший в буфер. Буфер заполняется по возрастанию: каждый выбранный элемент минимален среди оставшихся. При равенстве первым берётся элемент левого отрезка, поэтому слияние сохраняет исходный порядок равных элементов. После прохода буфер становится источником для следующего прохода.

### Пример: `[7, 2, 5, 2, 9, 1, 5]`

| Длина отрезка до слияния | Отсортированные отрезки после прохода |
| --- | --- |
| 1 | `[2, 7]`, `[2, 5]`, `[1, 9]`, `[5]` |
| 2 | `[2, 2, 5, 7]`, `[1, 5, 9]` |
| 4 | `[1, 2, 2, 5, 5, 7, 9]` |

В последнем слиянии сначала выбирается `1` из правой части, потом две `2` из левой. При сравнении `5` и `5` первой берётся левая пятёрка. После правой пятёрки остаются `7` и `9`.

### Сложность

**Время — O(n log n)** при `n ≥ 2`: каждый проход записывает `n` элементов, а длина отрезка удваивается. Первоначальное копирование занимает O(n). **Память — O(n)**: копия входа и один буфер длины `n`. Буфер переиспользуется; стек вызовов не растёт.

## `quick_sort_iterative`: разбиение со стеком диапазонов

Для текущего диапазона `[left, right)` опорным становится значение по индексу `(left + right) // 2`. Цикл с тремя указателями делит диапазон на элементы меньше, равные и больше опорного. Когда элемент меняется местами с последним ещё не обработанным элементом, указатель просмотра остаётся на месте: новое значение ещё предстоит сравнить.

Равные опорному элементы уже на своих местах. Из двух других отрезков меньший обрабатывается сразу, больший ждёт в стеке. Когда меньший отрезок исчерпан, цикл достаёт следующий диапазон из стека. После обработки всех диапазонов каждый элемент левой части предыдущего разбиения не больше любого элемента правой части, и все части отсортированы.

### Пример: `[7, 2, 5, 2, 9, 1, 5]`

Правая граница диапазона в таблице не входит в диапазон. Часть длиной ноль или один дальше не обрабатывается.

| Диапазон | Опорное | Меньше / равны / больше | Список после разбиения | Стек после разбиения |
| --- | --- | --- | --- | --- |
| `[0, 7)` | 2 | `[1]` / `[2, 2]` / `[9, 5, 5, 7]` | `[1, 2, 2, 9, 5, 5, 7]` | `[3, 7)` |
| `[3, 7)` | 5 | `[]` / `[5, 5]` / `[7, 9]` | `[1, 2, 2, 5, 5, 7, 9]` | `[5, 7)` |
| `[5, 7)` | 9 | `[7]` / `[9]` / `[]` | `[1, 2, 2, 5, 5, 7, 9]` | Пуст |

### Сложность

**Время — O(n log n) в среднем** на случайном порядке различных значений и **O(n²) в худшем случае**. Одно разбиение линейно по длине диапазона. Если опорный элемент раз за разом оказывается минимальным или максимальным, длина уменьшается лишь на один. Для списка одинаковых элементов достаточно одного разбиения: O(n).

**Память — O(n) с учётом копии входа, O(log n) без неё.** Разбиение переставляет элементы на месте в копии. При накоплении отложенных диапазонов алгоритм спускается в меньшую часть размером не более половины текущего. Поэтому в стеке одновременно не более O(log n) диапазонов, даже если разбиения неравномерны.

## Проверки

Тесты охватывают пустой список, один элемент, отсортированные и обратные данные, отрицательные числа, повторения и длину, не равную степени двойки. На списках до 64 элементов с фиксированным генератором результат сравнивается с `sorted`; большие случаи проверяют обратный порядок и одинаковые значения. Отдельно проверяется, что исходный список не изменился и возвращён новый объект.

```sh
python -m unittest -v
```

## Реализация

In [1]:
def merge_sort_iterative(values: list[int]) -> list[int]:
    """Сортировка слиянием снизу вверх. Входной список не меняется."""
    source = values.copy()
    size = len(source)
    if size < 2:
        return source

    buffer = [0] * size
    width = 1
    while width < size:
        for start in range(0, size, 2 * width):
            middle = min(start + width, size)
            end = min(start + 2 * width, size)
            left, right = start, middle

            for dest in range(start, end):
                if left < middle and (right == end or source[left] <= source[right]):
                    buffer[dest] = source[left]
                    left += 1
                else:
                    buffer[dest] = source[right]
                    right += 1

        source, buffer = buffer, source
        width *= 2

    return source


def quick_sort_iterative(values: list[int]) -> list[int]:
    """Быстрая сортировка со стеком диапазонов. Входной список не меняется."""
    result = values.copy()
    pending = [(0, len(result))]

    while pending:
        left, right = pending.pop()
        while right - left > 1:
            pivot = result[(left + right) // 2]
            lower = scan = left
            upper = right

            # [left, lower) < pivot, [lower, scan) == pivot,
            # [scan, upper) ещё не обработан, [upper, right) > pivot.
            while scan < upper:
                if result[scan] < pivot:
                    result[lower], result[scan] = result[scan], result[lower]
                    lower += 1
                    scan += 1
                elif result[scan] > pivot:
                    upper -= 1
                    result[scan], result[upper] = result[upper], result[scan]
                else:
                    scan += 1

            # Продолжаем с меньшей частью, большую откладываем в стек.
            if lower - left < right - upper:
                if right - upper > 1:
                    pending.append((upper, right))
                right = lower
            else:
                if lower - left > 1:
                    pending.append((left, lower))
                left = upper

    return result

## Пример использования

In [2]:
data = [7, 3, 5, 2, 5, -1, 4]
print(merge_sort_iterative(data))
print(quick_sort_iterative(data))

[-1, 2, 3, 4, 5, 5, 7]
[-1, 2, 3, 4, 5, 5, 7]


## Тесты

In [3]:
import random
import unittest



class TestIterativeSorts(unittest.TestCase):
    sorters = (merge_sort_iterative, quick_sort_iterative)

    def test_boundaries_and_nontrivial_inputs(self):
        cases = (
            ([], []),
            ([42], [42]),
            ([2, 1], [1, 2]),
            ([1, 2, 3, 4], [1, 2, 3, 4]),
            ([4, 3, 2, 1], [1, 2, 3, 4]),
            ([7, 2, 5, 2, 9, 1, 5], [1, 2, 2, 5, 5, 7, 9]),
            ([0, -7, 5, -7, 0, 2], [-7, -7, 0, 0, 2, 5]),
        )
        for sort in self.sorters:
            for values, expected in cases:
                with self.subTest(sort=sort.__name__, values=values):
                    snapshot = values.copy()
                    actual = sort(values)
                    self.assertEqual(actual, expected)
                    self.assertEqual(values, snapshot)
                    self.assertIsNot(actual, values)

    def test_against_sorted_on_seeded_inputs(self):
        generator = random.Random(2710813)
        for size in range(65):
            values = [generator.randint(-30, 30) for _ in range(size)]
            expected = sorted(values)
            for sort in self.sorters:
                with self.subTest(sort=sort.__name__, size=size):
                    self.assertEqual(sort(values), expected)

    def test_large_reverse_sorted_and_equal_inputs(self):
        cases = (
            (list(range(2048, 0, -1)), list(range(1, 2049))),
            ([3] * 2048, [3] * 2048),
        )
        for sort in self.sorters:
            for values, expected in cases:
                with self.subTest(sort=sort.__name__, first=values[0]):
                    self.assertEqual(sort(values), expected)

suite = unittest.TestSuite()
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(TestIterativeSorts))
report = unittest.TextTestRunner(verbosity=2).run(suite)
assert report.wasSuccessful(), 'Не все тесты прошли'

test_against_sorted_on_seeded_inputs (__main__.TestIterativeSorts.test_against_sorted_on_seeded_inputs) ... 

ok


test_boundaries_and_nontrivial_inputs (__main__.TestIterativeSorts.test_boundaries_and_nontrivial_inputs) ... 

ok


test_large_reverse_sorted_and_equal_inputs (__main__.TestIterativeSorts.test_large_reverse_sorted_and_equal_inputs) ... 

ok


----------------------------------------------------------------------
Ran 3 tests in 0.010s

OK
